# 10 — Product Search Smoke Test

Metadata retrieval in this version uses:

`FAISS rank + Tantivy BM25 rank → weighted RRF → lexical grounding`

سپس Grounded LLM reranker روی shortlist اجرا می‌شود.

ستون‌های `rrf_score`, `lexical_score`, `bm25_rank`,
`embedding_rank` و raw scoreها عمداً برای debug نمایش داده می‌شوند.

In [ ]:
from pathlib import Path
import sys
import os

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.preprocessing.processor import TextProcessor
from src.rag.embedding.factory import EmbeddingFactory
from src.rag.product_search import (
    ProductFAISSIndex,
    ProductBM25Index,
    ProductMetadataRetriever,
    ProductSearchReranker,
)
from src.rag.runtime import load_retrieval_stack
from src.rag.generation import OpenAIJSONGenerator
from src.rag.pipeline.product_search import ProductSearchPipeline

load_dotenv()

In [ ]:
rag_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "rag.yaml"
)

qa_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "qa.yaml"
)

search_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "product_search.yaml"
)

processor = TextProcessor()

embedding_model = (
    EmbeddingFactory.create(
        provider=rag_config[
            "embedding"
        ]["provider"],
        model_name=rag_config[
            "embedding"
        ]["model"],
    )
)

dense_index = (
    ProductFAISSIndex()
    .load(
        PROJECT_ROOT
        / "data"
        / "indexes"
        / "products_embedding"
    )
)

sparse_index = (
    ProductBM25Index(
        processor=processor
    )
    .load(
        PROJECT_ROOT
        / "data"
        / "indexes"
        / "products_bm25_tantivy"
    )
)

In [ ]:
metadata_cfg = (
    search_config[
        "product_search"
    ][
        "metadata"
    ]
)

metadata_retriever = (
    ProductMetadataRetriever(
        embedding_model=embedding_model,
        dense_index=dense_index,
        sparse_index=sparse_index,
        processor=processor,
        bm25_weight=metadata_cfg[
            "bm25_weight"
        ],
        embedding_weight=metadata_cfg[
            "embedding_weight"
        ],
        candidate_multiplier=metadata_cfg[
            "candidate_multiplier"
        ],
        brand_boost=metadata_cfg[
            "brand_boost"
        ],
        lexical_weight=metadata_cfg[
            "lexical_weight"
        ],
        rrf_k=metadata_cfg[
            "rrf_k"
        ],
        validate_index_alignment=metadata_cfg[
            "validate_index_alignment"
        ],
    )
)

review_stack = (
    load_retrieval_stack(
        project_root=PROJECT_ROOT
    )
)

In [ ]:
reranker_cfg = (
    search_config[
        "product_search"
    ][
        "reranker"
    ]
)

reranker_generator = (
    OpenAIJSONGenerator(
        api_key=os.getenv(
            "METIS_API_KEY"
        ),
        base_url=os.getenv(
            "METIS_BASE_URL"
        ),
        model=qa_config[
            "generation"
        ]["model"],
        input_cost_per_million=(
            qa_config[
                "generation"
            ].get(
                "input_cost_per_million"
            )
        ),
        output_cost_per_million=(
            qa_config[
                "generation"
            ].get(
                "output_cost_per_million"
            )
        ),
    )
)

reranker = (
    ProductSearchReranker(
        generator=reranker_generator,
        max_reviews_per_product=(
            reranker_cfg[
                "max_reviews_per_product"
            ]
        ),
        max_review_chars=(
            reranker_cfg[
                "max_review_chars"
            ]
        ),
    )
)

candidate_cfg = (
    search_config[
        "product_search"
    ][
        "candidates"
    ]
)

search = ProductSearchPipeline(
    metadata_retriever=metadata_retriever,
    review_retriever=review_stack.hybrid,
    reranker=reranker,
    metadata_candidates=(
        candidate_cfg[
            "metadata_products"
        ]
    ),
    review_comments_per_product=(
        candidate_cfg[
            "review_comments_per_product"
        ]
    ),
    reranker_candidates=(
        candidate_cfg[
            "reranker_products"
        ]
    ),
    metadata_weight=(
        reranker_cfg[
            "metadata_weight"
        ]
    ),
    reranker_weight=(
        reranker_cfg[
            "llm_weight"
        ]
    ),
)

In [ ]:
queries = [
    "ضد آفتاب پوست چرب که جوش نزنه",
    "شامپو ضد ریزش سریتا",
    "کرم آبرسان سبک و زود جذب",
]

for query in queries:
    print()
    print("QUERY:", query)

    results = search.search(
        query,
        top_k=10,
    )

    display(
        results[
            [
                "id",
                "title_fa",
                "Brand",
                "Category1",
                "Price",
                "score",
                "metadata_score",
                "rrf_score",
                "lexical_score",
                "token_overlap",
                "bigram_overlap",
                "bm25_rank",
                "embedding_rank",
                "bm25_raw_score",
                "embedding_raw_score",
                "brand_match",
                "llm_match_score",
                "evidence_status",
                "evidence_ids",
                "reason",
            ]
        ]
    )

    print(
        "Search telemetry:",
        results.attrs.get(
            "telemetry"
        )
    )